# snapshot

> Flat namespace serializer

In [ ]:
#| default_exp snapshot

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
from paar.snapshot import snapshot
def test_sorted_and_fields():
    rows = snapshot({'b':1, 'a':'hi'})
    assert [r.name for r in rows]==['a','b']
    assert rows[0].type=='str' and rows[0].value=="'hi'" and rows[0].path=='a'
    assert rows[1].type=='int' and rows[1].qualifier=='builtins'
def test_skips_hidden_and_dunder():
    rows = snapshot({'x':1, '_ih':2, '__y':3}, hidden=frozenset({'_ih'}))
    assert [r.name for r in rows]==['x']
def test_shape_populated():
    assert snapshot({'L':[1,2,3]})[0].shape=='3'
def test_flat_no_container():
    assert snapshot({'d':{'a':1}})[0].is_container is False
for t in [test_sorted_and_fields,test_skips_hidden_and_dunder,test_shape_populated,test_flat_no_container]: t()

In [ ]:
#| export
from paar.core import VarInfo
from paar.reprs import safe_repr, get_shape, get_dtype

def _var_info(name, v):
    "Build a flat VarInfo for one binding (P1: no container resolution)."
    try:
        return VarInfo(name=name, type=type(v).__name__,
                       qualifier=getattr(type(v), '__module__', '') or '',
                       value=safe_repr(v), shape=get_shape(v), dtype=get_dtype(v),
                       path=name)
    except Exception as e:
        return VarInfo(name=name, type='?', value=f'<error {e}>', is_error=True, path=name)

def snapshot(ns, hidden=frozenset()):
    "namespace dict -> sorted list[VarInfo], skipping hidden and dunder names."
    keys = sorted(k for k in ns if k not in hidden and not k.startswith('__'))
    return [_var_info(k, ns[k]) for k in keys]

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()